# 03 · Yeni sipariş simülasyonu ve önce/sonra karşılaştırması

Ürünün temel sorusu: *Bu siparişi eklersem ne zaman biter, hangi işler etkilenir?*

`core.scenarios` akışı:
1. `create_snapshot`: başlangıç planı çözülür, doğrulanır ve sürüm kimliğiyle dondurulur.
2. `simulate_order`: yeni sipariş **ayrı bir senaryoda** çözülür; başlangıç planı değişmez.
3. `compare_scenarios` (simülasyonun içinde): makespan, gecikme ve etkilenen işler hesaplanır.

> Sentetik demo; süreler ve terminler varsayımdır.

In [1]:
# Notebook'u tutorials/ veya repo kökünden açabilirsiniz; importlar repo kökünden çalışır.
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = "notebook_connected"   # plotly.js'i her notebook'a gömmez; görüntülemek için internet gerekir
print("Repo kökü:", ROOT)

Repo kökü: /Users/nazlsu/Desktop/PickWise


In [2]:
import pandas as pd

from benchmarks.workshop_order_demo import DemoInput, chart
from core.scenarios import OrderRequest, create_snapshot, simulate_order

demo = DemoInput.model_validate_json(Path("data/examples/order_demo.json").read_text(encoding="utf-8"))
baseline = create_snapshot(demo.baseline, "demo_baseline_v1", time_limit=10)
print("Başlangıç planı:", baseline.version_id, "|", baseline.result.status, "| makespan", baseline.result.makespan, "dk")
chart(baseline.scenario, baseline.result, "Önce: başlangıç planı")

Başlangıç planı: demo_baseline_v1 | OPTIMAL | makespan 15 dk


## Üç senaryo

| Senaryo | Yeni sipariş | Mevcut bitişler |
|---|---|---|
| Tek makine | yalnız Torna 1 (10 dk) | serbest |
| Alternatif makine | Torna 1 (10 dk) veya Torna 2 (12 dk) | serbest |
| Mevcut bitişleri koru | yalnız Torna 1 | önceki bitişlerden geç olamaz |

Yeni siparişin sert termini t=20'dir.

In [3]:
before_dump = baseline.model_dump()
simulations = {}
for i, (name, order, protect) in enumerate([
    ("Tek makine", demo.new_order, False),
    ("Alternatif makine", demo.alternative_order, False),
    ("Mevcut bitişleri koru", demo.new_order, True),
], start=1):
    simulations[name] = simulate_order(baseline, OrderRequest(
        baseline_version_id=baseline.version_id,
        scenario_id=f"workshop_demo_sim_{i}",
        order=order,
        preserve_existing_completions=protect,
        allow_demo=True,               # demo rotası; gerçek siparişte kullanıcı onaylı rota gerekir
        time_limit_seconds=10,
    ))

rows = []
for name, sim in simulations.items():
    c = sim.comparison
    rows.append({
        "Senaryo": name,
        "Durum": sim.result.status,
        "Makespan (dk)": None if c is None else f"{c.before.makespan} → {c.after.makespan}",
        "Geç biten mevcut iş": None if c is None else c.delayed_existing_job_count,
        "Toplam termin gecikmesi (dk)": None if c is None else c.after.total_tardiness,
        "Mesaj": sim.message,
    })
pd.DataFrame(rows).convert_dtypes()

,Senaryo,Durum,Makespan (dk),Geç biten mevcut iş,Toplam termin gecikmesi (dk),Mesaj
0,Tek makine,OPTIMAL,15 → 25,1,5,Bu model ve hedef için optimum plan kanıtlandı.
1,Alternatif makine,OPTIMAL,15 → 17,0,0,Bu model ve hedef için optimum plan kanıtlandı.
2,Mevcut bitişleri koru,INFEASIBLE,<NA>,<NA>,<NA>,Bu senaryonun modellenen kısıtları altında uyg...


**INFEASIBLE okuması:** "Mevcut bitişleri koru" senaryosunda modellenen kısıtlarla plan yok; bu yüzden karşılaştırma
uydurulmaz (`comparison = None`). Bu, yalnız *bu kısıtlar altında* imkânsızlıktır.

Simülasyonlar başlangıç planını değiştirmedi:

In [4]:
print("Başlangıç planı aynı mı:", baseline.model_dump() == before_dump)

Başlangıç planı aynı mı: True


## İş bazında etki

In [5]:
for name in ["Tek makine", "Alternatif makine"]:
    c = simulations[name].comparison
    print(name)
    display(pd.DataFrame([{
        "İş": i.name,
        "Önce bitiş": i.before_completion,
        "Sonra bitiş": i.after_completion,
        "Fark (dk)": i.completion_delta,
        "Sonra termin gecikmesi": "terminsiz" if i.after_tardiness is None else i.after_tardiness,
        "Değişen operasyon": i.changed_operations,
    } for i in c.impacts]))

Tek makine


,İş,Önce bitiş,Sonra bitiş,Fark (dk),Sonra termin gecikmesi,Değişen operasyon
0,Devam eden iş,10,10,0,0,0
1,Bekleyen mevcut iş,15,25,10,5,1


Alternatif makine


,İş,Önce bitiş,Sonra bitiş,Fark (dk),Sonra termin gecikmesi,Değişen operasyon
0,Devam eden iş,10,10,0,0,0
1,Bekleyen mevcut iş,15,15,0,0,0


In [6]:
best = simulations["Alternatif makine"]
chart(best.scenario, best.result, "Sonra: alternatif makine ile")

## Onaylı sürümü kayda hazırlamak

Kullanıcı yeni planı seçerse `core.plan_contract.StoredPlanVersion` ile denetim bilgisi eklenmiş bir kayıt nesnesi
oluşur. Bu nesne **veritabanına yazmaz**; SQLite uygulaması Yunus'un iş paketidir.

In [7]:
from datetime import datetime
from zoneinfo import ZoneInfo

from core.plan_contract import StoredPlanVersion
from core.scenarios import PlanSnapshot

new_version = PlanSnapshot(version_id="demo_v2", scenario=best.scenario, result=best.result)
record = StoredPlanVersion(
    plan_id="demo_atolye",
    parent_version_id=baseline.version_id,
    approved_by="nazlisu",
    approved_at=datetime(2026, 10, 17, 14, 0, tzinfo=ZoneInfo("Europe/Istanbul")),
    snapshot=new_version,
)
print(record.plan_id, record.parent_version_id, "→", record.snapshot.version_id)

demo_atolye demo_baseline_v1 → demo_v2


## Özet

- Simülasyon ayrı senaryodur; onaylı plan ancak kullanıcı seçince ayrı sürüm olur.
- Amaç makespan azaltmadır; yumuşak termin gecikmesi raporlanır, sert termin kısıttır.
- Komut satırı karşılığı: `python -m benchmarks.workshop_order_demo` → `results/workshop_order/comparison.html`.